# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [4]:
loan_complaint_data[0].page_content

"The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers."

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided context, the most common issues with loans appear to relate to mismanagement, inaccurate information, and difficulties in repayment. Specific frequent problems include errors in loan balances, incorrect reporting of account status, problems with how payments are being applied (often towards interest instead of principal), unauthorized transfers or changes in loan servicers, and lack of transparency or communication from lenders or servicers. Overall, these issues point to challenges with loan mishandling and miscommunication, which affect borrowers' ability to manage and understand their loans properly."

In [11]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints indicate that they were not handled in a timely manner. Specifically, there are complaints where the response was marked as "No" for timely response, such as the case involving MOHELA (Complaint ID: 12709087), where the consumer noted that the company took longer than the 15-day window to respond. Additionally, multiple complaints mention that the issue remains unresolved for extended periods, with delays over several weeks or months, suggesting that some complaints did not get handled promptly.'

In [12]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of factors highlighted in the complaints:\n\n1. **Lack of Clear Communication:** Borrowers were often inadequately notified about when their repayment was to resume, loan transfers, or changes in payment requirements, leading to unintentional delinquency and credit damage.\n\n2. **Complex and Predatory Payment Processes:** Many complaint narratives indicate that added funds are often applied in ways that primarily cover interest rather than reducing principal, making it difficult to pay off loans faster or reducing outstanding debt.\n\n3. **High Interest Accumulation:** For some borrowers, interest continued to accumulate during forbearance or deferment, which negated payments made and extended the total payoff period, often increasing the total amount owed.\n\n4. **Difficulty in Accessing or Changing Payment Plans:** Several borrowers experienced issues with loan servicers’ online systems, lack of available income-b

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to dealing with lenders or servicers, such as miscommunication, incorrect information, or difficulties in managing payments and loan details. Multiple complaints mention issues like incorrect application of payments, lack of clear or accurate loan information, and disputes about fees or loan terms.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, all the complaints listed were responded to with a "Closed with explanation," and the responses were marked as "Timely response?": "Yes." Therefore, it appears that any complaints mentioned did get handled in a timely manner.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for several reasons, including:\n\n1. Problems with payment plans and incorrect advice from servicers, such as being steered into wrong types of forbearances or having automatic payments disrupted without proper communication.\n2. Lack of proper communication from loan servicers, leading borrowers to be unaware of transfers, overdue status, or changes in their repayment arrangements.\n3. Difficulty resolving issues despite attempts to seek help, resulting in unpaid bills and delinquency.\n4. Mistakes or issues caused by the servicers, such as incorrect billing, failed payments, or mismanagement, which can negatively impact credit scores.\n5. Borrowers experiencing confusion or frustration due to inadequate information and administration errors that prevent timely repayment.\n\nIn summary, failures to repay loans often stem from mismanagement by loan servicers, insufficient communication, and administrative errors that hinder borrowers' ability to 

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

#### Answer #1:

"what is the interest rate on my student loan?"
in that query BM25 will prioritize documents with those exact terms ("interest" "rate" "student" "loan").

Or a query that has specific vocabulary like a company name and/or especific terminology, BM25 will gives higher ranks to documents where such words appear repeatedly.


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans, particularly student loans, appears to be dealing with the lender or servicer, often involving errors such as incorrect loan balances, misapplied payments, wrongful denials of payment plans, and mishandling of loan data. Many complaints also involve lack of clear communication, inaccurate information, and improper handling of personal data.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, there are indications that some complaints were not handled in a timely manner. For example, in one complaint, the individual reported that it has been over a year since they submitted a request and they have not received a response, and the issue has remained unresolved for nearly 18 months. Similarly, another complaint mentions a problem that has persisted over 2-3 weeks without resolution. Although the responses to these complaints were marked as "Timely response? Yes," the ongoing nature of some issues suggests that they were not resolved promptly from the complainants\' perspectives.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. Lack of Awareness: Borrowers were often unaware that they needed to repay their loans, especially if they were the first in their family to attend college and were not informed by financial aid officers.\n\n2. Poor Communication from Servicers: Borrowers reported receiving little to no notification about when payments were due, when their loans were transferred to new servicers, or how to set up payment plans, leading to missed payments and confusion.\n\n3. Difficulty Managing Payments: Many borrowers found the available repayment options, such as forbearance or deferment, led to accruing interest, which increased their total debt over time and made repayment more difficult.\n\n4. Financial Hardship: The economic realities of borrowers, including stagnant wages, inflation, and the burden of accumulating interest, made it hard to make payments without extending the payoff period or increasing total debt.\n\n5. 

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issues with loans, particularly student loans, include:\n\n- **Mismanagement by servicers**, such as errors in loan balances, misapplied payments, wrongful denials of repayment plans, and errors in loan balances.\n- **Problems with loan repayment handling**, including difficulty applying payments correctly, restrictions on paying down principal, and predatory repayment practices.\n- **Lack of transparency and poor communication**, such as not receiving timely notices about loan transfers, changes in servicers, or account status.\n- **Inaccurate or incomplete information**, including incorrect loan balances, misreported late payments, or loans belonging to someone else.\n- **Issues with loan validation and fraud concerns**, including disputes over the legitimacy of loans, identity theft, or insufficient documentation of loan origination and servicing rights.\n- **Problems with forgiveness, cancellation, or discharge**, often compounded by 

In [26]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints indicate that they did not get handled in a timely manner. Specifically:\n\n- Complaint #12709087 (Row 441) from CA, received on 03/28/25, was marked as "No" in response to whether it was handled timely.\n- Complaint #12739706 (Row 67) from NJ, received on 04/01/25, was marked as "No" for timely response.\n- Complaint #12654977 (Row 95) from MD, received on 03/25/25, was marked as "No".\n- Complaint #12973003 (Row 66) from PA, received on 04/14/25, was marked as "Yes", so it was handled in a timely manner.\n- Other complaints also reflect delays or failures to respond promptly, with some reports of exceeding expected response times and ongoing unresolved issues.\n\nTherefore, the answer is: Yes, some complaints did not get handled in a timely manner.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to issues related to mishandling and misconduct by loan servicers, lack of proper information about repayment options, and systemic practices that create obstacles to manageable repayment. Specific reasons include:\n\n1. **Misleading and Bad Information**: Borrowers reported receiving incorrect or incomplete information about their loans, repayment options, and consequences, which prevented them from making informed decisions.\n\n2. **Steering into Forbearance and Compounded Interest**: Many borrowers were repeatedly placed into long-term forbearances instead of available income-driven repayment plans or rehab programs. This practice led to accruing interest and ballooning balances, making repayment more difficult.\n\n3. **Coercive and Unfair Practices**: Several complaints describe servicers coercing borrowers into consolidations or forbearance, often without explaining available alternatives like income-driven plans that could redu

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

#### Answer #2:

Different users express the same intent using different words, a query "loan problems" might be reformulated as "loan complaints", "loan issues", etc.

There will be times where the user uses colloquial language while the documents use technical language, this will help overcome that

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be related to misconduct by loan servicers, including errors such as incorrect loan balances, misapplied payments, wrongful denials of payment plans, and issues with inaccurate or unverified debt reporting. Additionally, other prevalent issues include difficulties with loan reporting accuracy, unfair interest rate increases, and problems arising from loan transfers or sale of loans, leading to confusion and potential violations of consumer rights.'

In [34]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, according to the provided information, multiple complaints did not get handled in a timely manner. Specifically, complaints about delays and lack of response from Mohela (rows 441 and 84) were marked as "No" under the "Timely response?" field, indicating they were not handled promptly. Additionally, the complaint about the credit bureaus (row 474) was handled within the expected timeframe ("Yes" under "Timely response?"). However, the complaints related to Mohela clearly experienced issues with timely handling.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often fail to pay back their loans due to a variety of reasons highlighted in the complaints. Some common causes include:\n\n1. Lack of proper information or communication from loan servicers, such as not being notified about payment obligations or contact issues.\n2. Financial hardship or economic difficulties that make it impossible to meet repayment obligations.\n3. Misrepresentation or lack of transparency about the long-term consequences of taking out loans, leading to unforeseen financial burdens.\n4. Issues related to the quality and value of the education or institution, which can impact employment prospects and income, making loan repayment difficult.\n5. Problems with loan management, including delays, errors, or disputes over the legitimacy of the debt.\n6. Health problems or personal issues that hinder the ability to earn income and repay loans.\n\nIn summary, failure to repay loans can stem from communication failures, financial hardship, misrepresentation, or disp

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [37]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided data, the most common issues with loans appear to be related to mismanagement, misinformation, and handling problems by loan servicers. Specifically, the issues frequently reported include:\n\n- **Errors in loan balances and interest calculations** (e.g., balances growing unexpectedly, incorrect interest accrual)\n- **Dealing with the loan servicer or lender, including poor communication, lack of transparency, and inability to get accurate information**\n- **Problems with payment application, such as additional payments only going toward interest or being applied improperly**\n- **Incorrect loan classification or transfer mishandling, affecting deferments and loan status**\n- **Receiving bad or misleading information about loan terms, forgiveness eligibility, and account status**\n- **Inaccurate or unverified information impacting credit reports**\n\nWhile individual experiences vary, the recurring theme is that loan servicer misconduct, errors, and lack of trans

In [39]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints indicate that complaints were not handled in a timely manner. Specifically:\n\n- Complaint ID: 12739706 (row 127), received on 04/01/25, was marked as **"Timely response?": No**, indicating it was not handled promptly.\n- Complaint ID: 12935889 (row 129), received on 04/11/25, was marked as **"Timely response?": No**, also indicating delay.\n- Complaint ID: 12744910 (row 127), received on 03/31/25, was marked as **"Timely response?": Yes**, so handled promptly.\n- Additional complaints such as ID: 13062402 and 13070546 were timely, but others like ID 12739706 and 12935889 explicitly were not.\n\nOverall, there are multiple instances where complaints did not get handled in a timely manner, with at least two clearly marked as delayed.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans often due to a combination of factors highlighted in the complaints, including:\n\n- Lack of clear communication and notification from lenders or servicers about when repayment was to begin or changes in account status, leading to unintentional delinquency.\n- Difficulties in understanding or accessing information about their loans, interest accumulation, or repayment options, often compounded by errors or mismanagement by servicers.\n- The accumulation of interest during periods like deferment or forbearance, which can cause the total debt to grow, making repayment increasingly difficult or seemingly impossible.\n- Economic hardships such as unemployment, low income, or unexpected expenses, which make it hard for borrowers to afford their payments.\n- Issues with improper transfer or sale of loans without proper notice, resulting in missed payments or negative reporting.\n- In some cases, borrowers were steered into unfavorable payment plans or f

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [42]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [43]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [44]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [45]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [46]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints, the most common issues with loans appear to be related to loan servicing problems, such as trouble with repayment plans (e.g., incorrect payment amounts, faulty auto-debit setups, or re-amortization issues), poor communication from loan servicers, and issues with improper reporting or handling of loan status (including default, delinquency, or misinformation on credit reports). Additionally, some complaints involve illegal or improper collection and reporting practices, as well as breaches of privacy or data security.\n\nIn summary, the most common issues are:\n- Problems with repayment plans and payment processing\n- Lack of clear communication and transparency from loan servicers\n- Misinformation or errors on credit reports\n- Disputes over loan status, including default or delinquency\n- Unauthorized or illegal reporting and data breaches\n\nIf you need a specific aspect emphasized or further details, please let me know!'

In [47]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, based on the provided complaints, several complaints indicate that their issues were not handled in a timely manner. Specifically, the complaint from May 4, 2025, about transferred accounts involving misconduct from XXXX to Nelnet reflects that after multiple certified mailings detailing serious violations, Nelnet never responded to the complaint. Similarly, multiple complaints express ongoing issues with communication, incorrect billing, or unaddressed legal disputes, suggesting delays or failures in handling their complaints promptly. \n\nTherefore, the answer is: Yes, some complaints did not get handled in a timely manner.'

In [48]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues such as receiving bad information about their loans, difficulties with loan forgiveness documentation, problems with how payments were being handled (such as auto-debit issues, re-amortization delays, or payment processing errors), and disputes over the legitimacy or status of their loans. Additionally, some borrowers experienced challenges related to improper or illegal reporting of their loan status, unauthorized access to their personal information, and delays or stalls caused by loan servicers deliberately trying to hinder repayment or cause borrowers to give up. Overall, these issues often stem from lack of transparency, administrative errors, miscommunication, or legal complications involving loan servicers and government agencies.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?


#### Answer #3:

Highly repetitive sentences would have very similar embeddings, that will make it difficult for the chunker to indentify good breaking points.

This would help adjust the algorithm:

```python
   def adaptive_semantic_chunker(documents, embeddings):
       # calculate the avg sentence lenght and the repetition score
       avg_sentence_length = calculate_avg_sentence_length(documents)
       repetition_score = calculate_repetition_score(documents)
       
       if repetition_score > 0.8:  # only the most dissimilar sentences will create breakepoints
           threshold = 98  
       elif avg_sentence_length < 50:  # here the effect would be a combination of short sentences into meaningful chunks
           threshold = 90 
       else:
           threshold = 85 # defaults to 85, this is a balances approach
       
       return SemanticChunker(
           embeddings,
           breakpoint_threshold_type="percentile",
           percentile_threshold=threshold
       )
```

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

# golden dataset

In [ ]:
import pandas as pd
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
    ContextPrecision,
    ContextRecall,
    ContextRelevance,
    AnswerRelevancy,
    Faithfulness
)
from typing import List
import random

from ragas.llms import LangchainLLMWrapper

# existing model you already created earlier
chat_model = ChatOpenAI(model="gpt-4.1-nano")

# wrap it so ragas recognises it as a BaseRagasLLM
llm_for_evaluation = LangchainLLMWrapper(chat_model)

def create_golden_dataset(documents, num_questions=30):
    """Create synthetic questions and answers for evaluation."""
    
    # Sample questions based on common loan complaint themes
    sample_questions = [
        "What are the most common loan issues?",
        "How do customers typically complain about loan processing?",
        "What problems do people face with loan payments?",
        "What are the main complaints about loan companies?",
        "How do customers describe loan application problems?",
        "What issues arise with loan documentation?",
        "What are common complaints about loan terms?",
        "How do customers report loan service problems?",
        "What problems occur with loan refinancing?",
        "What are the main issues with loan customer service?",
        "How do customers describe loan approval delays?",
        "What problems arise with loan interest rates?",
        "What are common complaints about loan fees?",
        "How do customers report loan communication issues?",
        "What problems occur with loan modifications?",
        "What are the main issues with loan collections?",
        "How do customers describe loan account problems?",
        "What problems arise with loan disbursement?",
        "What are common complaints about loan agreements?",
        "How do customers report loan billing issues?",
        "What problems occur with loan insurance?",
        "What are the main issues with loan escrow?",
        "How do customers describe loan payment processing?",
        "What problems arise with loan statements?",
        "What are common complaints about loan customer support?",
        "How do customers report loan application status?",
        "What problems occur with loan verification?",
        "What are the main issues with loan underwriting?",
        "How do customers describe loan approval processes?",
        "What problems arise with loan documentation requests?",
        "What are the most frequent customer complaints?",
        "How do customers describe their loan experiences?",
        "What issues arise with loan account management?",
        "What are common problems with loan customer support?",
        "How do customers report loan processing delays?"
    ]
    
    # Sample answers that correspond to the questions
    sample_answers = [
        "Based on the loan complaint data, customers commonly report issues with loan processing delays, payment problems, and poor customer service.",
        "Customers typically complain about slow loan processing, unclear communication, and unexpected fees.",
        "Common payment problems include incorrect billing, payment processing errors, and difficulty making payments.",
        "Main complaints include poor customer service, lack of transparency, and unfair loan terms.",
        "Application problems include excessive documentation requests, long processing times, and unclear requirements.",
        "Documentation issues include excessive paperwork, unclear requirements, and lost documents.",
        "Common complaints about loan terms include high interest rates, hidden fees, and unfair conditions.",
        "Service problems include poor communication, unresponsive staff, and lack of support.",
        "Refinancing issues include high fees, complex processes, and unclear terms.",
        "Customer service problems include long wait times, unhelpful staff, and lack of resolution.",
        "Approval delays include long processing times, unclear status updates, and excessive requirements.",
        "Interest rate issues include unexpected increases, unclear calculations, and unfair rates.",
        "Fee complaints include hidden charges, excessive fees, and unclear fee structures.",
        "Communication issues include poor responses, unclear information, and lack of updates.",
        "Modification problems include complex processes, unclear requirements, and long delays.",
        "Collection issues include aggressive tactics, unclear processes, and unfair practices.",
        "Account problems include incorrect information, billing errors, and poor management.",
        "Disbursement issues include delays, incorrect amounts, and unclear processes.",
        "Agreement complaints include unclear terms, unfair conditions, and hidden clauses.",
        "Billing issues include incorrect charges, unclear statements, and payment problems.",
        "Insurance problems include unclear requirements, high costs, and poor coverage.",
        "Escrow issues include incorrect calculations, unclear processes, and poor management.",
        "Payment processing problems include delays, errors, and unclear procedures.",
        "Statement issues include unclear information, incorrect charges, and poor formatting.",
        "Support problems include long wait times, unhelpful staff, and lack of resolution.",
        "Application status issues include unclear updates, long delays, and poor communication.",
        "Verification problems include excessive requirements, unclear processes, and long delays.",
        "Underwriting issues include unclear criteria, long delays, and unfair decisions.",
        "Approval process problems include unclear requirements, long delays, and poor communication.",
        "Documentation request issues include excessive requirements, unclear processes, and poor communication.",
        "The most frequent complaints include poor customer service, processing delays, and billing issues.",
        "Customers describe experiences with unclear communication, long wait times, and unhelpful support.",
        "Account management issues include incorrect information, billing errors, and poor customer service.",
        "Common problems include long wait times, unhelpful staff, and lack of resolution.",
        "Processing delays include long wait times, unclear status updates, and poor communication."
    ]
    
    # Randomly sample questions and answers
    selected_indices = random.sample(range(len(sample_questions)), min(num_questions, len(sample_questions)))
    
    # Create evaluation dataset directly as DataFrame
    eval_dataset = pd.DataFrame({
        'question': [sample_questions[i] for i in selected_indices],
        'answer': [sample_answers[i] for i in selected_indices],
        'contexts': [[] for _ in range(len(selected_indices))]  # Will be filled by retrievers
    })
    
    return eval_dataset

# Create the golden dataset
golden_dataset = create_golden_dataset(loan_complaint_data, num_questions=30)
print(f"Created {len(golden_dataset)} evaluation questions")

Created 30 evaluation questions


# evaluation function

In [ ]:
# -------------------------------------------------------------
# 1. Build a proper Ragas EvaluationDataset
# -------------------------------------------------------------
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

def build_eval_dataset(df, ctxs):
    """df: DataFrame with columns question & answer
       ctxs: list[list[str]]  – contexts retrieved for every row
    """
    samples = []
    for i, row in df.iterrows():
        samples.append(
            SingleTurnSample(
                user_input        = row["question"],
                reference         = row["answer"],        # ground-truth answer
                retrieved_contexts= ctxs[i],              # list[str]
            )
        )
    return EvaluationDataset(samples=samples)

# -------------------------------------------------------------
# 2. Retrieve contexts once for every retriever
# -------------------------------------------------------------
def get_contexts(retriever, df):
    return [
        [doc.page_content for doc in retriever.get_relevant_documents(q)]
        for q in df["question"]
    ]

# -------------------------------------------------------------
# 3. Evaluate – use ONLY retrieval metrics (they need no `response`)
# -------------------------------------------------------------
from ragas.metrics import (
    ContextPrecision,
    ContextRecall,
    ContextRelevance,
)

def ragas_retriever_metrics(retriever_name, retriever, df, llm):
    """
    Returns average scores for precision / recall / relevance
    plus latency (s) and average #contexts.
    """
    import time, asyncio

    t0 = time.time()
    ctxs = get_contexts(retriever, df)
    latency = time.time() - t0

    eval_ds = build_eval_dataset(df, ctxs)

    # instantiate metrics with the LLM that Ragas will call
    metrics = [
        ContextPrecision(llm=llm_for_evaluation),
        ContextRecall(llm=llm_for_evaluation),
        ContextRelevance(llm=llm_for_evaluation),
    ]

    result = evaluate(eval_ds, metrics=metrics, llm=llm_for_evaluation,
                      show_progress=False)

    return {
        "retriever"         : retriever_name,
        "latency"           : latency,
        "context_precision" : result["context_precision"],
        "context_recall"    : result["context_recall"],
        "context_relevance" : result["context_relevance"],
        "avg_contexts"      : sum(map(len, ctxs)) / len(ctxs),
    }

# -------------------------------------------------------------
# 4. Run the evaluation loop
# -------------------------------------------------------------
retrievers = {
    "Naive Retrieval"      : naive_retriever,
    "BM25"                 : bm25_retriever,
    "Contextual Compression": compression_retriever,
    "Multi-Query"          : multi_query_retriever,
    "Parent Document"      : parent_document_retriever,
    "Ensemble"             : ensemble_retriever,
}

scores = []
for name, r in retrievers.items():
    print(f"Evaluating {name} …")
    try:
        scores.append(ragas_retriever_metrics(name, r, golden_dataset, chat_model))
        print("   ✓ done")
    except Exception as e:
        print(f"   ✗ failed → {e}")

import pandas as pd
result_df = pd.DataFrame(scores)
print("\n", result_df.to_string(index=False, float_format="%.3f"))

Evaluating Naive Retrieval …


KeyboardInterrupt: 